# 17_select_actives_for_dude — DUD-E 제출용 active 대표 선정

DUD-E 웹서버에 넣을 **active 대표 SMILES(≤50/제출)**를 고른다. DUD-E는 제출한 active마다
property-matched decoy(가정 inactive)를 생성하므로, inactive가 부족한 우리 데이터에
decoy를 늘리는 입력이 된다.

**배치 개념:** 1:5·1:10처럼 decoy가 많이 필요하면 45개씩 여러 번 제출한다. 배치마다 이전 배치와
**다른 active**를 넣어야 새로운 decoy가 생겨 화학공간을 넓게 덮는다. `BATCH`를 1,2,3...으로
바꿔 재실행하면 이전 배치를 자동 제외하고 새 45개를 뽑는다.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import glob
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator, DataStructs
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.SimDivFilters import rdSimDivPickers
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

### 파라미터

`BATCH`만 바꿔 재실행하면 배치별 파일(`data/dude_actives_batchN.smi`)이 쌓인다.

In [ ]:
# ===== 파라미터 =====
N_PER_BATCH = 45        # DUD-E 1회 제출 개수 (상한 50, 45 권장)
BATCH = 1               # 지금 생성할 배치 번호. 1:5·1:10 위해 2,3...으로 늘려가며 재실행
MW_MAX = 700.0          # DUD-E는 drug-like 위주 → 큰 분자 제외
SRC = 'data/HSD17B13_IC50_merged.xlsx'
OUT = f'data/dude_actives_batch{BATCH}.smi'
print(f'배치 {BATCH} | 배치당 {N_PER_BATCH}개 | 출력 {OUT}')

### active 선별 규칙

- IC50 ≤ 10000nM = active
- 염 제거(가장 큰 조각) 후 **InChIKey로 중복제거**(SMILES가 못 잡는 호변이성질체까지)
- DUD-E는 drug-like 위주 → **MW ≤ 700**만

In [ ]:
# active 선별: IC50<=10000=active. 큰 프래그먼트만(염 제거) + InChIKey 중복제거(호변이성질체)
df = pd.read_excel(SRC, sheet_name='same_dedup_keepdiff').dropna(subset=['canonical_smiles', 'ic50_nM'])

def klass(v, rel):
    rel = str(rel).strip()
    if rel in ('<', '<='): return 'active' if v <= 10000 else ('gray' if v < 20000 else 'inactive')
    if rel in ('>', '>='): return 'inactive' if v >= 20000 else ('gray' if v > 10000 else 'amb')
    return 'active' if v <= 10000 else ('gray' if v < 20000 else 'inactive')

lfc = rdMolStandardize.LargestFragmentChooser()
def clean(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return None
    m = lfc.choose(m)
    try: ik = Chem.MolToInchiKey(m)
    except Exception: return None
    return Chem.MolToSmiles(m), ik, Descriptors.MolWt(m)

rank = {'active': 0, 'gray': 1, 'inactive': 2, 'amb': 3}
best = {}
for smi, v, rel in zip(df.canonical_smiles, df.ic50_nM, df.relation):
    r = clean(smi)
    if r is None: continue
    cs, ik, mw = r
    c = klass(v, rel)
    if ik not in best or rank[c] < best[ik][0]:
        best[ik] = (rank[c], cs, mw, c)

acts = [(cs, ik, mw) for ik, (p, cs, mw, c) in best.items() if c == 'active' and mw and mw <= MW_MAX]
print(f'고유 active(InChIKey, MW<={MW_MAX:.0f}): {len(acts)}개')

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s, _, _ in acts]
ik_to_idx = {ik: i for i, (_, ik, _) in enumerate(acts)}

### 이전 배치 제외 (겹침 방지)

배치 2 이상이면 앞 배치들에서 이미 제출한 active를 빼서, 새 배치가 새로운 decoy를 만들게 한다.

In [ ]:
# 이전 배치(1..BATCH-1)에서 이미 제출한 active를 찾아 제외
used_idx = []
for b in range(1, BATCH):
    fn = f'data/dude_actives_batch{b}.smi'
    if not os.path.exists(fn): continue
    for line in open(fn):
        parts = line.split()
        if not parts: continue
        m = Chem.MolFromSmiles(parts[0])
        if m is None: continue
        ik = Chem.MolToInchiKey(m)
        if ik in ik_to_idx: used_idx.append(ik_to_idx[ik])
used_idx = sorted(set(used_idx))
print(f'이전 배치에서 이미 쓴 active: {len(used_idx)}개 → 이번 배치에서 제외')

### MaxMin 다양성 선정

서로 가장 다른 45개를 고른다. `firstPicks`에 이전 배치를 넣어 이전 배치와도 멀게 한다.

In [ ]:
# MaxMin 다양성 선정. firstPicks=이전배치 → 이전 배치와도 멀리 떨어진 새 45개
picker = rdSimDivPickers.MaxMinPicker()
def dist(i, j):
    return 1.0 - DataStructs.TanimotoSimilarity(fps[i], fps[j])

total = min(len(used_idx) + N_PER_BATCH, len(acts))
picked = list(picker.LazyPick(dist, len(fps), total, firstPicks=list(used_idx), seed=42))
used_set = set(used_idx)
new_idx = [i for i in picked if i not in used_set][:N_PER_BATCH]
print(f'이번 배치 선정: {len(new_idx)}개')

### DUD-E 형식 저장 & 출력

출력된 `SMILES 이름` 목록을 DUD-E 입력창에 붙여넣으면 된다.

In [ ]:
# DUD-E 형식 저장: 'SMILES  이름'(이름은 추적용, 공백 없이)
NL = chr(10)
with open(OUT, 'w') as f:
    for k, i in enumerate(new_idx, 1):
        name = 'b' + str(BATCH) + '_act_' + format(k, '02d')
        f.write(acts[i][0] + ' ' + name + NL)

sub = [fps[i] for i in new_idx]
nn = [max(DataStructs.TanimotoSimilarity(sub[a], sub[b]) for b in range(len(sub)) if a != b)
      for a in range(len(sub))]
print('저장:', OUT)
print('선정', len(new_idx), '개 서로간 최근접 Tanimoto 평균', round(float(np.mean(nn)), 2), '(낮을수록 다양)')
print()
print('=== DUD-E 제출용 (복사) ===')
for k, i in enumerate(new_idx, 1):
    print(acts[i][0], 'b' + str(BATCH) + '_act_' + format(k, '02d'))